# Sampling timing parameters with Discovery and Enterprise

This notebook is the everyday path: bind a `TimingSpec` to a pulsar, sample with Discovery (NumPyro) and Enterprise (PTMCMC), switch backends, and plot the chain.

`decentered_model` is the everyday Discovery sampler model; notebook 3 explains the alternative. Charts and geometry wait until notebooks 2 and 4.

A `TimingPulsar` is required. Today that is MetaPulsar; the `nltiming` calls do not change if Discovery or Enterprise grow a native host.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

import jax
import numpy as np
import matplotlib.pyplot as plt
import corner
import discovery as ds
from pathlib import Path
from metapulsar import create_metapulsar
from nltiming import TimingSpec, load_run
from nltiming.sampling import numpyro
from nltiming.sampling import ptmcmc

numpyro.ensure_x64()


## A TimingPulsar

AEI-DR2 combined J1721-2457 (isolated, 150 TOAs). Run this notebook from `examples/notebooks/`.


In [ ]:
DATA = Path("..") / "data" / "J1721-2457"
pulsar = create_metapulsar(
    {"combined": [{
        "par": DATA / "J1721-2457.par",
        "tim": DATA / "J1721-2457.tim",
        "timing_package": "tempo2",
    }]},
    combination_strategy="per_pta",
    use_pulse_numbers="reuse",
)
print(pulsar.name, len(pulsar.toas))


## Discovery

Default inference samples the nonlinear axes (here proper motion) and analytically marginalizes the rest. Short pedagogical chain — scale `num_warmup` / `num_samples` for science.


In [ ]:
spec = TimingSpec(engines="jug", name="timing")
timing = spec.for_pulsar(pulsar)
print("sampled:", timing.sampled)
print("marginalized:", timing.marginalized)

nd = {f"{pulsar.name}_efac": 1.0, f"{pulsar.name}_log10_t2equad": -8.0}
likelihood = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar, nd),
    *timing.discovery_signals(),
])
model = numpyro.decentered_model(likelihood, timing, fixed=nd)

from numpyro.infer import init_to_value
mcmc = numpyro.nuts(
    model, timing,
    num_warmup=200, num_samples=200, num_chains=1,
    init_strategy=init_to_value(values=numpyro.decentered_init_values(timing, model.transport)),
)
mcmc.run(jax.random.PRNGKey(0), extra_fields=numpyro.NUTS_EXTRA_FIELDS)


## Chains and a corner plot

`mcmc.to_df()` decodes sampler coordinates to physical columns `{pulsar}_timing_<fitpar>_theta_display`.


In [ ]:
df = mcmc.to_df()
cols = [c for c in df.columns if c.endswith("_theta_display")]
print(df[cols].describe())
corner.corner(df[cols].to_numpy(), labels=cols)
plt.show()

diag = numpyro.chain_diagnostics(mcmc)
print(diag["accept_prob"].mean(), diag["tree_depth_saturation_fraction"])


## Enterprise

Same default plan, libstempo backend. Enterprise parameters are prior-normal `z` (`..._timing_PMRA`), not physical units. Raw `PTSampler` — no `enterprise_extensions`.


In [ ]:
import tempfile
from enterprise.signals import parameter, signal_base, white_signals
from PTMCMCSampler.PTMCMCSampler import PTSampler

spec_ent = spec.with_engines({"tempo2": "libstempo"})
timing_ent = spec_ent.for_pulsar(pulsar)
white = white_signals.MeasurementNoise(efac=parameter.Uniform(0.5, 2.0))
pta = signal_base.PTA([(white + spec_ent.enterprise_signal())(pulsar)])
print(pta.param_names)

outdir = Path(tempfile.mkdtemp(prefix="nlt_ent_"))
timing_ent.write(
    outdir, likelihood="enterprise", sampler="ptmcmc",
    chain_layout=ptmcmc.chain_layout(timing_ent, pta.param_names),
)

x0 = np.hstack([np.asarray(p.sample(), dtype=float).reshape(-1) for p in pta.params])
sampler = PTSampler(
    len(pta.param_names), pta.get_lnlikelihood, pta.get_lnprior,
    np.diag(np.full(len(pta.param_names), 0.1**2)), outDir=str(outdir),
)
sampler.sample(x0, Niter=2000)

run = load_run(outdir)
post = run.posterior(burn=0.25)
corner.corner(np.column_stack([post[k] for k in timing_ent.sampled]), labels=list(timing_ent.sampled))
plt.show()


## Vela.jl

Eval only — Vela is not autodiff. PINT-native host.


In [ ]:
pulsar_pint = create_metapulsar(
    {"combined": [{
        "par": DATA / "J1721-2457.par",
        "tim": DATA / "J1721-2457.tim",
        "timing_package": "pint",
    }]},
    combination_strategy="per_pta",
    use_pulse_numbers="reuse",
)
timing_vela = TimingSpec(engines={"pint": "vela"}, name="timing").for_pulsar(pulsar_pint)
print(timing_vela.sampled)
print(np.std(pulsar_pint.residuals))
